Collecting real data from three sources – two API and one static web
page (via web scraping)


## 1. Imports and Configuration

We import the libraries needed for data collection.

- **requests** — sends HTTP requests to APIs and websites.
- **pandas** — works with tables and saves CSV files.
- **BeautifulSoup** — extracts data from HTML pages.
- **time** — adds delays between requests.
- **RobotFileParser** — checks website scraping rules.
- **product** — helps process Eurostat JSON data.

We also define 10 European regions with their NUTS 2 codes and city coordinates.

The `COLLECT_NEW_DATA` variable allows us to download new data or load existing CSV files.


In [ ]:
import time
from itertools import product
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
from bs4 import BeautifulSoup

COLLECT_NEW_DATA = True

HEADERS = {
    "User-Agent": "SIS-TourismProject/1.0 (contact: y_narbay@kbtu.com)"
}

REGIONS = {
    "FR10": ("Paris", 48.8566, 2.3522),
    "ES30": ("Madrid", 40.4168, -3.7038),
    "ES51": ("Barcelona", 41.3851, 2.1734),
    "DE30": ("Berlin", 52.5200, 13.4050),
    "AT13": ("Vienna", 48.2082, 16.3738),
    "EL30": ("Athens", 37.9838, 23.7275),
    "SE11": ("Stockholm", 59.3293, 18.0686),
    "CZ01": ("Prague", 50.0755, 14.4378),
    "ES70": ("Canarias", 28.1235, -15.4363),
    "NL32": ("Amsterdam", 52.3676, 4.9041)
}



## 2. Eurostat API — Tourism Data

We use the Eurostat API to collect official tourism statistics for 10 European NUTS 2 regions from 2022 to 2024.

The dataset `tour_occ_nin2m` contains monthly tourist overnight stays.

We send an HTTP GET request with parameters for regions, years, months, and accommodation types.

Eurostat returns data in JSON-stat format. We process it and convert the results into a pandas DataFrame.

The collected data is saved as `eurostat_nights_raw.csv`.

**Expected result:** 360 records (10 regions × 12 months × 3 years).


In [ ]:
#  Eurostat

if COLLECT_NEW_DATA:
    print("Downloading Eurostat data...")

    url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tour_occ_nin2m"

    params = {
        "format": "JSON",
        "lang": "EN",
        "freq": "A",
        "unit": "NR",
        "nace_r2": "I551-I553",
        "c_resid": "TOTAL",
        "geo": list(REGIONS),
        "time": ["2022", "2023", "2024"],
        "month": [f"M{m:02d}" for m in range(1, 13)]
    }

    try:
        response = requests.get(url, params=params, headers=HEADERS, timeout=40)
        response.raise_for_status()
        data = response.json()

        dimensions = data["id"]
        options = []

        for dim in dimensions:
            index = data["dimension"][dim]["category"]["index"]
            codes = sorted(index, key=index.get)
            options.append(codes)

        records = []

        for i, combination in enumerate(product(*options)):
            labels = dict(zip(dimensions, combination))

            year = labels["time"]
            month = labels["month"]
            nuts = labels["geo"]

            records.append({
                "NUTS_CODE": nuts,
                "City_Center": REGIONS[nuts][0],
                "YearMonth": f"{year}-{month[1:]}",
                "Nights_Count": data.get("value", {}).get(str(i)),
                "Status": data.get("status", {}).get(str(i))
            })

        df_eurostat = pd.DataFrame(records)
        df_eurostat.to_csv("eurostat_nights_raw.csv", index=False)

        print("Eurostat saved:", len(df_eurostat), "rows")

    except (requests.RequestException, ValueError, KeyError) as e:
        print("Eurostat error:", e)
        df_eurostat = pd.DataFrame()

else:
    df_eurostat = pd.read_csv("eurostat_nights_raw.csv")
    print("Eurostat loaded from CSV")



## 3. Open-Meteo API — Historical Weather Data

We use the Open-Meteo Historical Archive API to collect daily weather data from January 2022 to December 2024.

For each region, we use the coordinates of a representative city.

We collect three weather variables:

- `temperature_2m_max` — daily maximum temperature (°C).
- `precipitation_sum` — daily precipitation (mm).
- `sunshine_duration` — daily sunshine duration (seconds).

We use a loop to download data for all 10 regions and combine the results into one DataFrame.

The data is saved as `weather_archive_raw.csv`.

**Expected result:** 10,960 daily records.


In [ ]:
#  Open-Meteo

if COLLECT_NEW_DATA:
    print("\nDownloading weather data...")

    url = "https://archive-api.open-meteo.com/v1/archive"
    weather_tables = []

    for nuts, (city, lat, lon) in REGIONS.items():
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": "2022-01-01",
            "end_date": "2024-12-31",
            "daily": "temperature_2m_max,precipitation_sum,sunshine_duration",
            "timezone": "auto"
        }

        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=40)
            response.raise_for_status()

            daily = response.json()["daily"]

            df_city = pd.DataFrame(daily)
            df_city["NUTS_CODE"] = nuts
            df_city["City"] = city

            weather_tables.append(df_city)

            print(city, "-", len(df_city), "days")

        except (requests.RequestException, ValueError, KeyError) as e:
            print("Weather error:", city, e)

        time.sleep(1)

    if weather_tables:
        df_weather = pd.concat(weather_tables, ignore_index=True)
        df_weather.to_csv("weather_archive_raw.csv", index=False)

        print("Weather saved:", len(df_weather), "rows")
    else:
        df_weather = pd.DataFrame()

else:
    df_weather = pd.read_csv("weather_archive_raw.csv")
    print("Weather loaded from CSV")



## 4. Wikipedia Web Scraping — City Rankings

We use web scraping to collect international visitor rankings from Wikipedia.

First, we check the website's `robots.txt` rules. Then we use `requests` to download the HTML page.

We use BeautifulSoup to find HTML tables and extract the ranking year, city name, and rank.

We also add a delay between requests to reduce the load on the website.

The extracted data is saved as `wiki_ranks_raw.csv`.

These rankings provide additional information about tourism in major cities.


In [ ]:
#  Wikipedia

if COLLECT_NEW_DATA:
    print("\nScraping Wikipedia...")

    url = "https://en.wikipedia.org/wiki/List_of_cities_by_international_visitors"
    df_wiki = pd.DataFrame()

    try:
        robots = requests.get(
            "https://en.wikipedia.org/robots.txt",
            headers=HEADERS,
            timeout=20
        )
        robots.raise_for_status()

        parser = RobotFileParser()
        parser.parse(robots.text.splitlines())

        if parser.can_fetch(HEADERS["User-Agent"], url):
            time.sleep(1)

            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, "html.parser")
            tables = soup.select("table.wikitable")

            rows = []

            for table in tables:
                heading = table.find_previous(["h2", "h3"])
                title = heading.get_text(" ", strip=True) if heading else ""

                if "2024" in title:
                    year = 2024
                    city_column = 1
                elif "2023" in title:
                    year = 2023
                    city_column = 1
                elif "2016/2018" in title:
                    year = 2018
                    city_column = 2
                else:
                    continue

                for tr in table.select("tr"):
                    cells = tr.find_all(["td", "th"], recursive=False)

                    if len(cells) < 3:
                        continue

                    rank = cells[0].get_text(" ", strip=True)
                    city = cells[city_column].get_text(" ", strip=True)

                    if rank.isdigit():
                        rows.append({
                            "Year": year,
                            "Rank": int(rank),
                            "City": city,
                            "Source_URL": url
                        })

            df_wiki = pd.DataFrame(rows)

            if not df_wiki.empty:
                df_wiki.to_csv("wiki_ranks_raw.csv", index=False)
                print("Wikipedia saved:", len(df_wiki), "rows")
            else:
                print("No ranking tables found")

        else:
            print("Scraping not allowed by robots.txt")

    except (requests.RequestException, ValueError) as e:
        print("Wikipedia error:", e)

else:
    df_wiki = pd.read_csv("wiki_ranks_raw.csv")
    print("Wikipedia loaded from CSV")



## 5. Data Collection Results

After collecting the data, we check the size of each dataset using `DataFrame.shape`.

We collected:

- **Eurostat:** 360 monthly tourism records.
- **Open-Meteo:** 10,960 daily weather records.
- **Wikipedia:** 120 city ranking records.

All three datasets are saved as separate CSV files for further processing.


In [ ]:
print("\n Data Collection Results ")
print("Eurostat:", df_eurostat.shape)
print("Weather:", df_weather.shape)
print("Wikipedia:", df_wiki.shape)

if not df_eurostat.empty and len(df_eurostat) != 360:
    print("Warning: Eurostat data may be incomplete")

if not df_weather.empty and len(df_weather) != 10960:
    print("Warning: Weather data may be incomplete")


## 6. Data Validation

We validate the collected data before passing it to the next team member.

We check:

- The total number of records.
- The number of unique regions.
- Missing values in each dataset.
- Duplicate records.
- The number of monthly records for each region.

Eurostat contains 360 records for 10 regions, with 36 months per region.

Open-Meteo contains 10,960 daily records for the same 10 regions.

No missing values or duplicate records were found in the main datasets.

Wikipedia contains 120 records without missing values.

These checks help us identify problems before data cleaning and merging.


In [ ]:
print("\n Data Validation ")

if not df_eurostat.empty:
    print("Eurostat rows:", len(df_eurostat))
    print("Regions:", df_eurostat["NUTS_CODE"].nunique())
    print("Missing nights:", df_eurostat["Nights_Count"].isna().sum())

    duplicates = df_eurostat.duplicated(
        subset=["NUTS_CODE", "YearMonth"]
    ).sum()

    print("Duplicate records:", duplicates)

    print("\nRecords by region:")
    print(df_eurostat.groupby("NUTS_CODE").size())

if not df_weather.empty:
    print("\nWeather rows:", len(df_weather))
    print("Regions:", df_weather["NUTS_CODE"].nunique())
    print("\nMissing values:")
    print(df_weather.isna().sum())

    duplicates = df_weather.duplicated(
        subset=["NUTS_CODE", "time"]
    ).sum()

    print("Duplicate weather records:", duplicates)

if not df_wiki.empty:
    print("\nWikipedia rows:", len(df_wiki))
    print("Missing values:")
    print(df_wiki.isna().sum())


Eurostat saved: 360 rows

Paris - 1096 days
Madrid - 1096 days
Barcelona - 1096 days
Berlin - 1096 days
Vienna - 1096 days
Athens - 1096 days
Stockholm - 1096 days
Prague - 1096 days
Canarias - 1096 days
Amsterdam - 1096 days
Weather saved: 10960 rows

Scraping Wikipedia...
Wikipedia saved: 120 rows

=== Data Collection Results ===
Eurostat: (360, 5)
Weather: (10960, 6)
Wikipedia: (120, 4)

=== Data Validation ===
Eurostat rows: 360
Regions: 10
Missing nights: 0
Duplicate records: 0

Records by region:
NUTS_CODE
AT13    36
CZ01    36
DE30    36
EL30    36
ES30    36
ES51    36
ES70    36
FR10    36
NL32    36
SE11    36
dtype: int64

Weather rows: 10960
Regions: 10

Missing values:
time                  0
temperature_2m_max    0
precipitation_sum     0
sunshine_duration     0
NUTS_CODE             0
City                  0
dtype: int64
Duplicate weather records: 0

Wikipedia rows: 120
Missing values:
Year          0
Rank          0
City          0
Source_URL    0
dtype: int64
